In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# EINLESEN

In [ ]:
insurance = pd.read_csv("insurance.csv",sep=";",skiprows=7)

insurance.head()

- Fehlende Werte sind leere Zellen --> wäre handelbar für Pandas

# EXPLORATION

In [ ]:
_ = insurance.hist(bins=50, figsize=(20,15))
plt.show()

In [ ]:
insurance.plot(kind="box", subplots=True, layout=(2, 2), figsize=(14, 8), sharex=False, sharey=False)
plt.tight_layout()
plt.show()

In [ ]:
# Kategorische Spalten auswählen
categorical_cols = insurance.select_dtypes(include=['object', 'category']).columns

fig, axes = plt.subplots(nrows=len(categorical_cols), ncols=1, figsize=(20, 5 * len(categorical_cols)))

# Falls nur eine kategorische Spalte vorhanden ist
if len(categorical_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, categorical_cols):
    insurance[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_ylabel('Anzahl')
    ax.tick_params(axis='x', rotation=0)

plt.show()

- 2 Ausreißer bei age
- die hohen Werte bei charges sind durchaus realistisch
- keine Sentinel Werte erkennbar.

- Generell: Was hat mit Sicherheit hohen Einfluss auf die charges? region könnte hohen Einfluss haben, aber auch Sex/ smoker

# TRAIN-TEST-SPLIT

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42) # Objekt erschaffen

for train_index, test_index in splitter.split(insurance, insurance["region"]):
    insurance_train = insurance.loc[train_index].reset_index(drop=True) # sortiert Datenreihen in Trainingsdaten und formatiert Index
    insurance_test  = insurance.loc[test_index].reset_index(drop=True) # sortiert Datenreihen in Testdaten und formatiert Index

In [ ]:
# Prüfen der Verteilung

print(f"Gesamt {insurance['region'].value_counts(normalize=True)}\n")
print(f"Train {insurance_train['region'].value_counts(normalize=True)}\n")
print(f"Test {insurance_test['region'].value_counts(normalize=True)}\n")

print(f"Gesamt {insurance['sex'].value_counts(normalize=True)}\n")
print(f"Train {insurance_train['sex'].value_counts(normalize=True)}\n")
print(f"Test {insurance_test['sex'].value_counts(normalize=True)}\n")

print(f"Gesamt {insurance['smoker'].value_counts(normalize=True)}\n")
print(f"Train {insurance_train['smoker'].value_counts(normalize=True)}\n")
print(f"Test {insurance_test['smoker'].value_counts(normalize=True)}\n")

In [ ]:
# Entfernen der Duplikate
print("Duplikate:", insurance_train.duplicated().any())                    # Erkennen Duplikaten
insurance_train = insurance_train.drop_duplicates().reset_index(drop=True) # Datenreihen mit fehlenden Werten werden rausgeworfen

In [ ]:
# Trennung von Features und Zielvariablen
X_insurance_train = insurance_train.drop("charges", axis=1)   # Features
y_insurance_train = insurance_train["charges"].copy()         # Zielvariable

In [ ]:
# Trennung von numerischen und kategoriale Spalten
cat_cols = ["sex", "smoker", "region"]
X_insurance_train_num = X_insurance_train.drop(cat_cols, axis=1)
X_insurance_train_cat = X_insurance_train[cat_cols]

# PIPELINE

In [ ]:
# Behandlung von Ausreißern
class IQRTransformer(BaseEstimator, TransformerMixin):
    """Beschränkt Werte auf IQR-basierte Grenzen (Clipping). NaN-Werte bleiben erhalten."""

    def fit(self, X, y=None):                                        # y=None, damit Aufruf im Transformer immer funktioniert 
        self.quantiles_ = np.nanpercentile(X, [25.0, 75.0], axis=0)  # Berechnet Q1 und Q3 und speichert in Attribut self.quantiles
        self.iqr_ = self.quantiles_[1] - self.quantiles_[0]          # Berechnet den IQR (Q3 - Q1) und speichert in Attribut self.iqr
        return self

    def transform(self, X):
        X_transformed = np.copy(X)
        for feature in range(X.shape[1]):
            if np.isnan(self.iqr_[feature]):
                continue
            lower_bound = self.quantiles_[0, feature] - 1.5 * self.iqr_[feature]  # wird auf Grenzwert unten gesetzt
            upper_bound = self.quantiles_[1, feature] + 1.5 * self.iqr_[feature]  # wird auf Grenzwert oben gesetzt
            X_transformed[:, feature] = np.where(
                np.isnan(X_transformed[:, feature]),
                X_transformed[:, feature],
                np.clip(X_transformed[:, feature], lower_bound, upper_bound)
            )
        return X_transformed

In [ ]:
# Aufbau der Pipeline
# Daten werden als Numpy Arrays weitergegeben

num_pipeline = Pipeline(steps=[
    # ('-1_to_NaN', NegativeOneToNaNTransformer()),
    ("iqr_outlier", IQRTransformer()),                     # Behandlung von Ausreißern
    ("imputer",     SimpleImputer(strategy="median")),     # Behandlung fehlender Werte
    # ('attribs_adder', CombinedAttributesAdder(add_bedrooms_per_room=True)),
    ("std_scaler",  StandardScaler()),                     # Skalierung und Normalisierung - muss immer ans Ende!
])

cat_pipeline = Pipeline(steps=[
    ("1hot", OneHotEncoder()),
])                                                         # Encodierung mit 1/0 in numerische Spalten


In [ ]:
# Spalten werden transformiert
num_attribs = list(X_insurance_train_num.columns)
cat_attribs = list(X_insurance_train_cat.columns)    # Listen mit den Spaltennamen werden erstellt

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),      # die numerischen Spaltenbezeichnungen
    ("cat", cat_pipeline, cat_attribs),      # die kategorischen Spaltenbezeichnungen
])

X_insurance_train_prepared = full_pipeline.fit_transform(X_insurance_train)


In [ ]:
# DataFrame mit beschreibenden Spaltennamen erstellen
cat_attribs_names = (
    full_pipeline
    .named_transformers_["cat"]["1hot"]
    .get_feature_names_out(cat_attribs)                   # gibt die neuen Spaltenbezeichnungen für Kategorien zurück
)
all_attribs_names = num_attribs + list(cat_attribs_names) # Zusammenführung der Spaltenbezeichnungen zu einer Liste 

X_insurance_train_prepared = pd.DataFrame(                # Umwandlung in DataFrame
    X_insurance_train_prepared,
    columns=all_attribs_names                             # Überschreiben der generischen Spaltenbezeichnungen
)

print(X_insurance_train_prepared.shape)
X_insurance_train_prepared